# UK Economic Data — Exploratory Data Analysis

**Author:** Vishal Joshi | MSc Applied AI, University of Warwick

This notebook explores the UK macroeconomic time series used in the forecasting pipeline. We examine:

1. **Time series plots** of all major indicators
2. **Correlation analysis** between variables
3. **Stationarity tests** (Augmented Dickey-Fuller)
4. **Seasonality decomposition** for retail sales and industrial production
5. **Missing data analysis**
6. **Feature importance discussion** and modelling choices

---

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the Python path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yaml
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

from src.data.preprocessor import Preprocessor

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

%matplotlib inline

# Load configuration
with open(PROJECT_ROOT / 'config' / 'config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print(f'Project root: {PROJECT_ROOT}')

## 1. Load and Inspect Data

In [ ]:
# Load sample data through the preprocessor
preprocessor = Preprocessor(config, PROJECT_ROOT)
df = preprocessor.load_sample_data()

print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nData types:\n{df.dtypes}')
df.head(10)

In [ ]:
# Summary statistics
df.describe().round(2)

## 2. Time Series Plots

Visualise each major economic indicator over the sample period (2015-2024). Key events to look for:
- **2016**: Brexit referendum and its aftermath
- **2020**: COVID-19 pandemic — dramatic drops in GDP, retail, production
- **2021-2022**: Post-pandemic recovery and inflationary surge
- **2022-2023**: Bank of England rate-hiking cycle

In [ ]:
indicators = {
    'gdp': ('GDP Growth (Quarterly, Interpolated)', '%'),
    'cpi': ('CPI Inflation (Annual Rate)', '%'),
    'unemployment_rate': ('Unemployment Rate', '%'),
    'bank_rate': ('Bank of England Base Rate', '%'),
    'retail_sales_index': ('Retail Sales Index', 'Index'),
    'industrial_production_index': ('Industrial Production Index', 'Index'),
    'm4_growth': ('M4 Money Supply Growth', '%'),
    'mortgage_approvals': ('Mortgage Approvals', 'Thousands'),
}

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, (col, (title, unit)) in enumerate(indicators.items()):
    ax = axes[i]
    ax.plot(df.index, df[col], linewidth=1.5, color=sns.color_palette()[i % 8])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(unit)
    ax.axvspan('2020-03-01', '2020-06-01', alpha=0.15, color='red', label='COVID-19')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())

fig.suptitle('UK Macroeconomic Indicators (2015-2024)', fontsize=16, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

### Observations

- **GDP** shows the dramatic COVID-19 collapse in Q2 2020 (~-19.5% quarterly) followed by a V-shaped recovery.
- **CPI inflation** was subdued until 2021, then surged to over 11% in October 2022 — the highest in 40 years — driven by energy prices and supply-chain disruptions.
- **Unemployment** spiked during COVID but was cushioned by the furlough scheme, keeping the peak at ~5.2%.
- **Bank Rate** was slashed to 0.10% during COVID, then raised aggressively from December 2021 to combat inflation, reaching 5.25% by August 2023.
- **Retail sales** show strong seasonality (December spikes) and a COVID-related crash in April 2020.
- **Mortgage approvals** collapsed during lockdowns, briefly surged due to the stamp duty holiday, then fell sharply when rates rose.

These dynamics create a challenging but realistic forecasting problem.

## 3. Correlation Analysis

In [ ]:
# Correlation heatmap
corr = df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(
    corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
    annot=True, fmt='.2f', square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}, ax=ax,
)
ax.set_title('Correlation Matrix — UK Economic Indicators', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

### Key Correlations

- **CPI and Bank Rate** are strongly positively correlated — the BoE raises rates to combat inflation.
- **M4 money supply** and CPI show a time-lagged relationship — monetary expansion precedes price increases.
- **Mortgage approvals** and Bank Rate are negatively correlated — higher rates reduce housing demand.
- **GDP** and unemployment show the expected inverse (Okun's Law) relationship.

These cross-variable relationships justify using multivariate models (XGBoost, neural nets) alongside univariate ARIMA.

## 4. Stationarity Tests (ADF)

In [ ]:
# Augmented Dickey-Fuller test for each series
adf_results = []

for col in df.columns:
    series = df[col].dropna()
    result = adfuller(series, autolag='AIC')
    adf_results.append({
        'Variable': col,
        'ADF Statistic': round(result[0], 4),
        'p-value': round(result[1], 4),
        'Lags Used': result[2],
        'Critical 5%': round(result[4]['5%'], 4),
        'Stationary (5%)': 'Yes' if result[1] < 0.05 else 'No',
    })

adf_df = pd.DataFrame(adf_results)
adf_df.style.applymap(
    lambda v: 'color: green' if v == 'Yes' else ('color: red' if v == 'No' else ''),
    subset=['Stationary (5%)']
)

### Stationarity Discussion

Several series are non-stationary at the 5% level — particularly the Bank Rate, CPI, and indices that trend over time. This is expected for macroeconomic data and has modelling implications:

- **ARIMA** handles non-stationarity through differencing (the `d` parameter). `auto_arima` will select the appropriate order.
- **XGBoost** is invariant to monotonic transformations and handles non-stationary data natively.
- **Neural networks** receive standardised inputs (via `StandardScaler`), which partially addresses scale issues but not unit roots.

For robustness, we include both level values and differenced features (MoM/YoY changes) in the feature matrix.

## 5. Seasonal Decomposition

In [ ]:
# Seasonal decomposition for retail sales
retail = df['retail_sales_index'].dropna()
decomp_retail = seasonal_decompose(retail, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
decomp_retail.observed.plot(ax=axes[0], title='Observed')
decomp_retail.trend.plot(ax=axes[1], title='Trend')
decomp_retail.seasonal.plot(ax=axes[2], title='Seasonal')
decomp_retail.resid.plot(ax=axes[3], title='Residual')
fig.suptitle('Seasonal Decomposition — Retail Sales Index', fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Seasonal decomposition for industrial production
indprod = df['industrial_production_index'].dropna()
decomp_indprod = seasonal_decompose(indprod, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
decomp_indprod.observed.plot(ax=axes[0], title='Observed')
decomp_indprod.trend.plot(ax=axes[1], title='Trend')
decomp_indprod.seasonal.plot(ax=axes[2], title='Seasonal')
decomp_indprod.resid.plot(ax=axes[3], title='Residual')
fig.suptitle('Seasonal Decomposition — Industrial Production Index', fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

### Seasonality Observations

- **Retail sales** exhibit clear annual seasonality — a sharp spike in November-December (holiday shopping) followed by a January dip. This justifies using SARIMA with `m=12` and including `month` as a feature.
- **Industrial production** shows weaker but present seasonality, with dips in summer months and year-end.
- The COVID-19 shock in 2020 creates large residuals that disrupt the seasonal pattern — this is a challenge for all models.

## 6. Missing Data Analysis

In [ ]:
# Check raw sample files for missing data patterns
import glob

sample_dir = PROJECT_ROOT / 'data' / 'sample'
csv_files = sorted(sample_dir.glob('*.csv'))

missing_info = []
for f in csv_files:
    raw = pd.read_csv(f)
    missing_info.append({
        'File': f.name,
        'Rows': len(raw),
        'Missing Values': raw.isna().sum().sum(),
        'Date Range': f'{raw["date"].min()} to {raw["date"].max()}',
    })

pd.DataFrame(missing_info)

### Missing Data Strategy

The sample data is complete, but in production (API-sourced data), missingness arises because:

1. **Different publication frequencies** — GDP is quarterly, most others are monthly.
2. **Publication lags** — ONS releases data 4-6 weeks after the reference period.
3. **API gaps** — occasional missing observations due to data revisions.

Our strategy:
- **Quarterly to monthly**: Cubic spline interpolation (preserves growth shape).
- **Sporadic missing values**: Forward-fill (last known value is the best estimate).
- **Leading NaNs**: Back-fill from the first available observation.

This is documented in `src/data/preprocessor.py`.

## 7. Feature Importance and Modelling Choices

### Why Three Models?

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **ARIMA** | Interpretable, well-understood for time series, captures autoregressive dynamics | Univariate only, linear, struggles with structural breaks |
| **XGBoost** | Handles non-linearity, uses cross-variable information, robust to outliers | Not designed for sequential data, requires careful feature engineering |
| **Neural Net** | Flexible function approximator, can learn complex interactions | Data-hungry (our dataset is small), prone to overfitting, less interpretable |

### Feature Engineering Rationale

- **Lag features (t-1, t-3, t-6, t-12)**: Capture autoregressive behaviour. The 12-month lag is important for annual patterns.
- **Rolling statistics (3m, 6m, 12m)**: Smooth out noise and capture trends. Standard deviations measure volatility.
- **MoM/YoY changes**: Isolate momentum from level effects. Partially address non-stationarity.
- **Spread features**: The real interest rate (Bank Rate - CPI) is a key macro variable.
- **Interaction terms**: Non-linear relationships, e.g., unemployment*GDP captures Okun's Law dynamics.

### Honest Assessment

Economic forecasting is genuinely difficult. Our models will likely show:
- Reasonable 1-month-ahead forecasts (persistence is a strong baseline)
- Deteriorating accuracy at 3-month and 6-month horizons
- Inability to predict structural breaks (COVID, energy crisis)
- ARIMA may outperform complex models on smooth periods
- XGBoost should show the best overall balance

In [ ]:
# Cross-correlation analysis: which variables lead/lag each other?
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

pairs = [
    ('m4_growth', 'cpi', 'M4 Growth vs CPI (M4 leads inflation?)'),
    ('bank_rate', 'mortgage_approvals', 'Bank Rate vs Mortgage Approvals'),
    ('unemployment_rate', 'gdp', 'Unemployment vs GDP'),
    ('bank_rate', 'cpi', 'Bank Rate vs CPI'),
]

for ax, (var1, var2, title) in zip(axes.flatten(), pairs):
    lags = range(-12, 13)
    correlations = [df[var1].corr(df[var2].shift(lag)) for lag in lags]
    ax.bar(lags, correlations, color='steelblue', alpha=0.7)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Lag (months, positive = var1 leads)')
    ax.set_ylabel('Correlation')

fig.suptitle('Cross-Correlation Analysis', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

## Summary

This EDA confirms that:

1. The UK economic data contains rich structure — trends, seasonality, cross-variable correlations, and structural breaks.
2. Non-stationarity is prevalent, justifying the use of differencing (ARIMA) and change-based features (XGBoost/NN).
3. The 2020 COVID shock and 2022 inflation crisis create significant modelling challenges.
4. A multi-model approach is warranted — no single model dominates across all conditions.
5. Feature engineering should exploit both autoregressive and cross-variable signals.

These findings inform the design of the forecasting pipeline in `src/`.